In [53]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder , OrdinalEncoder , StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [37]:
df=pd.read_csv('titanic.csv')
df.sample(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
688,689,0,3,"Fischer, Mr. Eberhard Thelander",male,18.0,0,0,350036,7.7958,NaN,S
490,491,0,3,"Hagland, Mr. Konrad Mathias Reiersen",male,NaN,1,0,65304,19.9667,NaN,S
861,862,0,2,"Giles, Mr. Frederick Edward",male,21.0,1,0,28134,11.5000,NaN,S


In [38]:
df.isnull().sum()

,0
PassengerId,0
Survived,0
Pclass,0
Name,0
Sex,0
Age,177
SibSp,0
Parch,0
Ticket,0
Fare,0


In [41]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin','SibSp','Parch'],inplace=True)

KeyError: "['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp', 'Parch'] not found in axis"

In [42]:
df.sample(7)

,Survived,Pclass,Sex,Age,Fare,Embarked
388,0,3,male,NaN,7.7292,Q
788,1,3,male,1.0,20.5750,S
296,0,3,male,23.5,7.2292,C
497,0,3,male,NaN,15.1000,S
131,0,3,male,20.0,7.0500,S
479,1,3,female,2.0,12.2875,S
434,0,1,male,50.0,55.9000,S


In [43]:
x_train,x_test,y_train,y_test=train_test_split(df.drop(columns=['Survived']),df['Survived'],test_size=0.2,random_state=34)

In [44]:
tr1=ColumnTransformer(transformers=[
    ('tr1',SimpleImputer(),[3]),
    ('tr2',StandardScaler(),[3,4])
],remainder='passthrough')


tr2=ColumnTransformer(transformers=[
    ('tr3',OneHotEncoder(handle_unknown='ignore',sparse_output=False),[2,5])
],remainder='passthrough')


tr3=LogisticRegression()



In [45]:
pipe=Pipeline(steps=[
    ('tr1',tr1),
    ('tr2',tr2),
    ('tr3',tr3)
])

In [73]:

numerical_features = ['Age', 'Fare']
categorical_features = ['Sex', 'Embarked', 'Pclass']

numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])


preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features),
    ],
    remainder='passthrough')


pipe = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(solver='liblinear', random_state=34))
])

pipe.fit(x_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['Age', 'Fare']),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore',
                                                                                 sparse_output=False))]),
                                                  ['Sex', 'Embarked',
                                                   'Pclass'])])),
                ('classifier',
                 LogisticRegression(random_state=34, solver='liblinear'))])

In [74]:
pipe.score(x_test,y_test)

0.8156424581005587

In [76]:
new_sample = pd.DataFrame([[1,'male',24,17.234,'Q']], columns=x_train.columns)
ans=pipe.predict(new_sample)

if ans[0]==1:
    print('Survived')
else:
    print('Not Survived')

Survived


In [77]:
import pickle

pickle.dump(pipe,open('pipe.pkl','wb'))